# HRC Safety Log — Hybrid Safety AI (revised)

Notebook này triển khai pipeline kết hợp **luật khoảng cách cứng**, `ExtraTreesClassifier` và `IsolationForest`. Bản điều chỉnh dùng cách chia train/test theo `source_file` để tránh rò rỉ dữ liệu giữa các mẫu liên tiếp, đồng thời dùng confidence của ExtraTrees để giảm cảnh báo nhầm.

Ưu tiên quyết định: **EmergencyStop theo luật cứng → Warning theo khoảng cách → AI có confidence → SAFE**.


In [ ]:
from pathlib import Path
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import ExtraTreesClassifier, IsolationForest
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, f1_score,
    precision_score, recall_score
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Tự tìm feature_engineering.py để chạy được trên Kaggle Dataset
input_root = Path("/kaggle/input")
feature_files = list(input_root.rglob("feature_engineering.py"))
if feature_files:
    AI_MODEL_DIR = feature_files[0].parent
    ROOT = AI_MODEL_DIR.parent
else:
    ROOT = Path.cwd()
    AI_MODEL_DIR = ROOT / "ai_model"

sys.path.insert(0, str(ROOT))
from ai_model.feature_engineering import load_recordings, FEATURE_COLUMNS

DATA_DIR = AI_MODEL_DIR / "data" / "raw"
WARNING_CM = 60.0
DANGER_CM = 30.0
AI_CONFIDENCE = 0.70

print("ROOT:", ROOT)
print("DATA_DIR:", DATA_DIR)
print("Data exists:", DATA_DIR.exists())


In [ ]:
data = load_recordings(DATA_DIR)

print("Shape:", data.shape)
print(data["label"].value_counts())
print(data[["label", "label_id"]].drop_duplicates().sort_values("label"))

X = data[FEATURE_COLUMNS].copy()
y = data["label_id"].astype(int)
y_binary = (y > 0).astype(int)


In [ ]:
# Chia theo file ghi âm, tránh để các mẫu liên tiếp của cùng file
# xuất hiện đồng thời trong train và test.
groups = data["source_file"]
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train files:", data.iloc[train_idx]["source_file"].nunique())
print("Test files:", data.iloc[test_idx]["source_file"].nunique())
print("Test labels:")
print(y_test.value_counts().sort_index())


In [ ]:
# 1) ExtraTrees: phân loại có giám sát
classifier = ExtraTreesClassifier(
    n_estimators=400,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
classifier.fit(X_train, y_train)
classifier_pred = classifier.predict(X_test).astype(int)
classifier_proba = classifier.predict_proba(X_test)
classifier_confidence = classifier_proba.max(axis=1)

print("ExtraTrees classification report:")
print(classification_report(
    y_test, classifier_pred,
    labels=[0, 1, 2],
    target_names=["SAFE", "APPROACHING", "DANGER"],
    zero_division=0,
))


In [ ]:
# 2) Isolation Forest: chỉ học vùng bình thường
normal_train = X_train[y_train == 0]
anomaly_model = Pipeline([
    ("scale", StandardScaler()),
    (
        "detector",
        IsolationForest(
            n_estimators=300,
            contamination=0.10,
            random_state=42,
            max_features=1.0,
        ),
    ),
])
anomaly_model.fit(normal_train)
anomaly_pred = (anomaly_model.predict(X_test) == -1).astype(int)
actual_binary = (y_test > 0).astype(int).to_numpy()

print("IsolationForest binary report:")
print(classification_report(
    actual_binary, anomaly_pred,
    labels=[0, 1],
    target_names=["SAFE", "ANOMALY"],
    zero_division=0,
))


In [ ]:
# 3) Hybrid decision
def hybrid_predict(features, classifier, anomaly_model,
                   warning_cm=60.0, danger_cm=30.0,
                   confidence_threshold=0.70):
    features = features.reset_index(drop=True).copy()
    classifier_state = classifier.predict(features).astype(int)
    classifier_confidence = classifier.predict_proba(features).max(axis=1)
    anomaly_flag = (anomaly_model.predict(features) == -1).astype(int)

    states = []
    emergency_flags = []
    reasons = []

    for i, row in features.iterrows():
        minimum = float(row["min_distance_cm"])

        # Lớp an toàn cao nhất: không phụ thuộc AI
        if minimum <= danger_cm:
            states.append("EMERGENCY")
            emergency_flags.append(True)
            reasons.append("HARD_DANGER_THRESHOLD")

        elif minimum <= warning_cm:
            states.append("WARNING")
            emergency_flags.append(False)
            reasons.append("HARD_WARNING_THRESHOLD")

        # Ngoài vùng 60 cm, chỉ cảnh báo khi AI đủ tin cậy
        elif (classifier_state[i] > 0 and
              classifier_confidence[i] >= confidence_threshold):
            states.append("WARNING")
            emergency_flags.append(False)
            reasons.append("EXTRATREES_CONFIDENT")

        # Hoặc hai mô hình cùng đồng ý có bất thường
        elif classifier_state[i] > 0 and anomaly_flag[i] == 1:
            states.append("WARNING")
            emergency_flags.append(False)
            reasons.append("AI_AGREEMENT")

        else:
            states.append("SAFE")
            emergency_flags.append(False)
            reasons.append("NORMAL")

    return pd.DataFrame({
        "state": states,
        "emergency_stop": emergency_flags,
        "reason": reasons,
        "classifier_state": classifier_state,
        "classifier_confidence": classifier_confidence,
        "anomaly_flag": anomaly_flag,
    })

hybrid = hybrid_predict(
    X_test, classifier, anomaly_model,
    WARNING_CM, DANGER_CM, AI_CONFIDENCE
)

hybrid_binary = hybrid["state"].isin(["WARNING", "EMERGENCY"]).astype(int).to_numpy()

print("Hybrid binary report:")
print(classification_report(
    actual_binary, hybrid_binary,
    labels=[0, 1],
    target_names=["SAFE", "ANOMALY"],
    zero_division=0,
))

cm = confusion_matrix(actual_binary, hybrid_binary)
metrics = {
    "accuracy": float(accuracy_score(actual_binary, hybrid_binary)),
    "balanced_accuracy": float(balanced_accuracy_score(actual_binary, hybrid_binary)),
    "precision_anomaly": float(precision_score(actual_binary, hybrid_binary, zero_division=0)),
    "recall_anomaly": float(recall_score(actual_binary, hybrid_binary, zero_division=0)),
    "f1_anomaly": float(f1_score(actual_binary, hybrid_binary, zero_division=0)),
    "false_positive_rate_safe": float(cm[0, 1] / max(cm[0].sum(), 1)),
    "false_negative_rate_anomaly": float(cm[1, 0] / max(cm[1].sum(), 1)),
    "confusion_matrix": cm.tolist(),
}

print("Metrics:")
print(json.dumps(metrics, indent=2))


In [ ]:
# Vẽ confusion matrix hybrid
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["SAFE", "ANOMALY"],
)
fig, ax = plt.subplots(figsize=(6, 5))
display.plot(ax=ax, cmap="Blues", values_format="d", colorbar=True)
ax.set_title("Revised Hybrid Pipeline — Confusion Matrix")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.savefig("/kaggle/working/hybrid_confusion_matrix_revised.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Lưu artifact dùng cho gateway
artifact = {
    "classifier": classifier,
    "anomaly_model": anomaly_model,
    "feature_columns": FEATURE_COLUMNS,
    "thresholds_cm": {
        "warning": WARNING_CM,
        "danger": DANGER_CM,
    },
    "ai_confidence_threshold": AI_CONFIDENCE,
    "training_rows_classifier": int(len(X_train)),
    "training_rows_anomaly": int(len(normal_train)),
    "architecture": "hard_thresholds + confidence_ExtraTrees + IsolationForest_agreement",
}

OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_DIR.mkdir(exist_ok=True)
joblib.dump(artifact, OUTPUT_DIR / "hybrid_model_revised.joblib")
with open(OUTPUT_DIR / "hybrid_metrics_revised.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

print("Đã lưu:", OUTPUT_DIR / "hybrid_model_revised.joblib")
print("Đã lưu:", OUTPUT_DIR / "hybrid_metrics_revised.json")


## Kết luận

Bản điều chỉnh giảm cảnh báo nhầm bằng ba cơ chế: chia dữ liệu theo file ghi âm, chỉ dùng cảnh báo AI ngoài vùng 60 cm khi ExtraTrees đủ confidence hoặc hai mô hình cùng đồng ý, và giữ luật cứng cho EmergencyStop. Cần báo cáo riêng accuracy, recall anomaly, false-positive rate và false-negative rate; không chỉ báo cáo accuracy.
